# GW170817 Gravitational-Wave Data Analysis
## A Historic Binary Neutron-Star Merger

On August 17, 2017, LIGO-Virgo detected gravitational waves from a binary neutron-star merger and observed electromagnetic counterparts for the first time, opening a new era of multi-messenger astronomy.

In [ ]:
conda install -c conda-forge gwpy

In [ ]:
pip install --prefer-binary gwpy

In [ ]:
# Install required libraries
!pip install gwpy matplotlib scipy numpy

In [4]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.io.wavfile import write
from gwpy.timeseries import TimeSeries
from gwpy.signal import filter_design
from gwpy.plot import Plot


## 1. Data Acquisition

In [ ]:
# Set GW170817 event parameters
t0 = 1187008882.4  # event time (GPS time)
start_time = 1187008842  # start time (t0 - 40seconds)
end_time = 1187008922    # end time (t0 + 40seconds)

print(f"Event time: {t0}")
print(f"Data time range: {start_time} to {end_time}")
print(f"Duration: {end_time - start_time} seconds")

In [ ]:
# Fetch Hanford detector data from GWOSC
print("Downloading LIGO Hanford data from GWOSC...")
hdata = TimeSeries.fetch_open_data('H1', start_time, end_time)

print(f"Data acquisition complete!")
print(f"Number of samples: {len(hdata)}")
print(f"Sample rate: {hdata.sample_rate.value} Hz")
print(f"Duration: {hdata.duration.value} seconds")
print(f"Strain data range: {hdata.min():.2e} to {hdata.max():.2e}")

## 2. Spectral Analysis - Amplitude Spectral Density (ASD)

In [ ]:
# Calculate amplitude spectral density
print("Calculating amplitude spectral density...")
asd = hdata.asd()

# Plot the ASD curve
plt.figure(figsize=(12, 6))
plt.plot(asd.frequencies, asd, 'b-', linewidth=1, label='LIGO Hanford ASD')
plt.xlim(10, 2000)
plt.ylim(1e-24, 1e-19)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency [Hz]', fontsize=12)
plt.ylabel('Amplitude Spectral Density [strain/√Hz]', fontsize=12)
plt.title('GW170817 - LIGO Hanford Amplitude Spectral Density', fontsize=14)
plt.grid(True, alpha=0.3)

# Mark power-line interference
for freq in [60, 120, 180]:
    plt.axvline(freq, color='red', linestyle='--', alpha=0.5, linewidth=1)
    plt.text(freq, 2e-24, f'{freq}Hz', rotation=90, fontsize=10)

plt.legend()
plt.tight_layout()
plt.show()

## 3. Signal Filtering

In [ ]:
# Designing filters
print("Designing filters...")

# Bandpass filter: 50-250 Hz, preserving the characteristic gravitational-wave frequency band
bp = filter_design.bandpass(50, 250, hdata.sample_rate)

# Notch filters: remove power-line interference
notches = [filter_design.notch(line, hdata.sample_rate) for line in (60, 120, 180)]

# Combine all filters
zpk = filter_design.concatenate_zpks(bp, *notches)

print("Filter design complete:")
print(f"  Bandpass range: 50-250 Hz")
print(f"  Notch frequencies: 60, 120, 180 Hz")

In [ ]:
# Apply filters
print("Applying filters...")
hfilt = hdata.filter(zpk, filtfilt=True)

# Crop data edges to reduce edge effects
hdata_cropped = hdata.crop(*hdata.span.contract(1))
hfilt_cropped = hfilt.crop(*hfilt.span.contract(1))

print("Filtering complete!")
print(f"Raw data range: {hdata.span}")
print(f"Cropped range: {hdata_cropped.span}")

## 4. Filter Visualization

In [ ]:
# Plot the before-and-after filtering comparison
plot = Plot(
    hdata_cropped, 
    hfilt_cropped,
    figsize=[12, 8],
    separate=True,
    sharex=True,
    color='green'
)

ax1, ax2 = plot.axes

ax1.set_title('GW170817 - LIGO Hanford Strain Data (Before/After Filtering)', fontsize=14)
ax1.text(0.02, 0.95, 'Raw Data', transform=ax1.transAxes, fontsize=12, 
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax1.set_ylabel('Strain Amplitude', fontsize=12)

ax2.text(0.02, 0.95, 'Filtered Data (50-250Hz Bandpass + 60/120/180Hz Notches)', 
         transform=ax2.transAxes, fontsize=12,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax2.set_ylabel('Strain Amplitude', fontsize=12)
ax2.set_xlabel('Time [GPS seconds]', fontsize=12)

plot.show()

## 5. Spectral Comparison

In [ ]:
# Compare spectra before and after filtering
plt.figure(figsize=[12, 8])

# Calculate ASD
asd_original = hdata_cropped.asd()
asd_filtered = hfilt_cropped.asd()

# Plot the comparison
plt.plot(asd_original.frequencies, asd_original, 
         label='Original Data', color='red', alpha=0.7, linewidth=2)
plt.plot(asd_filtered.frequencies, asd_filtered, 
         label='Filtered Data', color='blue', alpha=0.7, linewidth=2)

plt.xlim(10, 2000)
plt.ylim(1e-24, 1e-19)
plt.xlabel('Frequency [Hz]', fontsize=12)
plt.ylabel('Amplitude Spectral Density [strain/√Hz]', fontsize=12)
plt.title('GW170817 - Spectrum Comparison: Before vs After Filtering', fontsize=14)

# Mark important regions
plt.axvspan(50, 250, alpha=0.1, color='green', label='Bandpass Region (50-250 Hz)')

for freq in [60, 120, 180]:
    plt.axvline(freq, linestyle="--", color="orange", alpha=0.7, linewidth=1)

plt.legend(loc='upper right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.tight_layout()
plt.show()

## 6. Signal Details Near the Event

In [ ]:
# Focus on the key moment around the event
plot = hfilt_cropped.plot(color='darkgreen', figsize=[12, 5])
ax = plot.gca()

# Set the display range: 20 seconds around the event
ax.set_xlim(t0 - 10, t0 + 10)
ax.set_xscale('seconds', epoch=t0)  # Use the event time as the zero point

ax.set_ylabel('Strain Amplitude', fontsize=12)
ax.set_title('GW170817 - Filtered Strain Data (Around Event)', fontsize=14)

# Mark the event time
ax.axvline(0, color='red', linestyle='-', alpha=0.7, linewidth=2, label='Merger Time')
ax.legend()

plot.show()

# Extract data for later analysis
x_val = plt.gca().lines[0].get_xdata()
y_val = plt.gca().lines[0].get_ydata()

## 7. Multi-Detector Comparison

In [ ]:
# Fetch Livingston detector data
print("Downloading LIGO Livingston data...")
ldata = TimeSeries.fetch_open_data('L1', start_time, end_time)

# Apply the same filters
lfilt = ldata.filter(zpk, filtfilt=True)
lfilt = lfilt.crop(*lfilt.span.contract(1))

# Shift L1 in time and adjust phase
lfilt.shift('6.9ms')  # Compensate for the signal arrival-time difference
lfilt *= -1  # Adjust phase because the detectors have different orientations

print("Livingston data processing complete!")

In [ ]:
# Plot the two-detector comparison
plot = Plot(figsize=[12, 6])
ax = plot.gca()

ax.plot(hfilt_cropped, label='LIGO Hanford (H1)', color='green', linewidth=1.5)
ax.plot(lfilt, label='LIGO Livingston (L1)', color='orange', linewidth=1.5)

ax.set_xlim(t0 - 10, t0 + 10)
ax.set_xscale('seconds', epoch=t0)
ax.set_ylabel('Strain Amplitude', fontsize=12)
ax.set_ylim(-1e-21, 1e-21)
ax.set_title('GW170817 - Dual Detector Data Comparison', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plot.show()

## 8. Q-Transform Time-Frequency Analysis

In [ ]:
# Run Q-transform analysis
print("Running Q-transform time-frequency analysis...")

# Focus on 10 seconds around the event because neutron-star merger signals are longer
hq = hfilt_cropped.q_transform(outseg=(t0-10, t0+10))
plot = hq.plot(figsize=[12, 6])

ax = plot.gca()
plot.colorbar(label="Normalized Energy", ax=ax)
ax.grid(False)
ax.set_yscale('log')
ax.set_ylim(20, 1000)
ax.set_title('GW170817 - Time-Frequency Analysis (Q-transform)', fontsize=14)
ax.set_ylabel('Frequency [Hz]', fontsize=12)
ax.set_xlabel('Time [seconds]', fontsize=12)

plot.show()

## 9. Gravitational-Wave Audio Generation

In [ ]:
# Convert the gravitational-wave signal into audio
print("Generating gravitational-wave audio...")

# Extract data within 1.5 seconds of the event
time_window = 1.5
ind = np.where((x_val < (t0 + time_window)) & (x_val > (t0 - time_window)))
y_audio = y_val[ind]

# Normalize the signal
y_audio = y_audio / np.max(np.abs(y_audio))

# Calculate sample rate
fs = int(1 / np.median(np.diff(x_val[ind])))

# Generate the WAV file
amplitude = np.iinfo(np.int16).max
audio_data = (y_audio * amplitude).astype(np.int16)

filename = "GW170817_gravitational_wave.wav"
write(filename, fs, audio_data)

print(f"Audio file generated: {filename}")
print(f"Sample rate: {fs} Hz")
print(f"Duration: {len(audio_data)/fs:.2f} seconds")
print(f"Number of samples: {len(audio_data)}")

# Plot the audio signal
plt.figure(figsize=(12, 4))
time_audio = np.linspace(-time_window, time_window, len(audio_data))
plt.plot(time_audio, audio_data, 'purple', linewidth=1)
plt.axvline(0, color='red', linestyle='--', alpha=0.7, label='Merger Time')
plt.xlabel('Time [seconds]', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.title('GW170817 - Gravitational Wave Audio Signal', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()